# 🌳 Step 6: Forest Density & Land Cover Layers
Classifies landscape into VDF, MDF, OF, Nurseries, and non-forest covers.

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

CLASSES = {
    1: "Very Dense Forest (>70%)", 2: "Moderately Dense Forest (40-70%)",
    3: "Open Forest (10-40%)", 4: "Sparse Forest (0-10%)",
    5: "Nursery / Saplings", 6: "Barren Land / Grassland",
    7: "No Forest / Built-up", 8: "Forest Office / Infrastructure"
}
COLORS = ['#004d00', '#008000', '#33cc33', '#99ff99', '#ffff00', '#d2b48c', '#ff0000', '#808080']
cmap = ListedColormap(COLORS)
print("✅ Layers Defined Successfully.")

In [ ]:
def generate_density_layers(ndvi_array, output_path):
    h, w = ndvi_array.shape
    density_map = np.zeros((h, w), dtype=np.uint8)
    density_map[ndvi_array > 0.7] = 1
    density_map[(ndvi_array >= 0.4) & (ndvi_array <= 0.7)] = 2
    density_map[(ndvi_array >= 0.1) & (ndvi_array < 0.4)] = 3
    density_map[(ndvi_array > 0.0) & (ndvi_array < 0.1)] = 4
    density_map[(ndvi_array <= 0.0) & (ndvi_array > -0.1)] = 6
    density_map[ndvi_array <= -0.1] = 7
    density_map[50:70, 50:70] = 5
    density_map[150:160, 150:160] = 8
    profile = {'driver': 'GTiff', 'height': h, 'width': w, 'count': 1, 'dtype': 'uint8', 'crs': 'EPSG:4326', 'transform': rasterio.transform.from_bounds(0, 0, 1, 1, w, h)}
    with rasterio.open(output_path, 'w', **profile) as dst: dst.write(density_map, 1)
    return density_map

dummy_ndvi = np.random.uniform(-0.2, 0.9, (200, 200))
result_map = generate_density_layers(dummy_ndvi, 'density_layers.tif')
print("\n✅ Density Layers Generated.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.imshow(result_map, cmap=cmap, interpolation='nearest')
handles = [plt.Rectangle((0,0),1,1, color=COLORS[k-1], label=f"{k}: {v}") for k, v in CLASSES.items()]
ax.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, title="Classification Layers")
ax.set_title("Forest Density & Land Cover Classification Map", fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()